# Round 8 final evaluation

Reopen this notebook from disk, restart the kernel, and run from the top. This is the single active notebook, using the original scripts and output folders.

The setup cell uses a separate Conda environment matching A's recorded versions. The next cell preserves the failed development report and cache in an archive, then the original paths are reused for fresh reproduction. No failed result is changed to a pass and the acceptance criteria remain unchanged.

Results: `results/round8_final/`. Caches: `artifacts/round8_final/`.

Codex has not installed packages or executed this notebook. Continue beyond development reproduction only if it passes.

In [ ]:
from pathlib import Path
import ast
import hashlib
import json
import os
import shutil
import subprocess
import sys

project = Path(r"C:\sc4001\prjJ\SC4001-AI-Image-Detection")
output = project / "results" / "round8_final"
env_prefix = project / "artifacts" / "round8_final" / "environments" / "a_recorded_runtime"
assert project.is_dir()
env = dict(os.environ, PYTHONDONTWRITEBYTECODE="1", PYTHONIOENCODING="utf-8",
           CUBLAS_WORKSPACE_CONFIG=":4096:8")
env.pop("PYTHONHOME", None)
env.pop("PYTHONPATH", None)
environment_ready = False

candidates = [
    os.environ.get("CONDA_EXE"),
    str(Path(sys.prefix) / "Scripts" / "conda.exe"),
    str(Path(sys.base_prefix) / "Scripts" / "conda.exe"),
    shutil.which("conda.exe"),
    shutil.which("conda"),
]
conda = next((p for p in candidates if p and Path(p).is_file()), None)
assert conda, "Conda was not found. Open this notebook from Anaconda/Jupyter, then restart the kernel."

def stream(command):
    print("Running:", subprocess.list2cmdline([str(p) for p in command]), flush=True)
    with subprocess.Popen(command, cwd=project, env=env,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, encoding="utf-8", errors="replace", bufsize=1) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    if code:
        raise RuntimeError(f"Command stopped with exit code {code}. Do not run later cells.")

def worker_command(*args):
    return [conda, "run", "--no-capture-output", "--prefix", str(env_prefix),
            "python", "-B", "-u", *args]

def run(script, *args):
    assert environment_ready, "Run the environment setup/check cell first."
    stream(worker_command(str(project / "scripts" / script), *args))

scripts = sorted((project / "scripts").glob("round8_final_*.py"))
assert len(scripts) == 7, [p.name for p in scripts]
for path in scripts:
    ast.parse(path.read_text(encoding="utf-8"), filename=str(path))
print("Syntax parsed:", len(scripts), "files.")
print("Jupyter kernel stays:", sys.executable)
print("Worker environment:", env_prefix)
print("Results:", output)

## 0. Create and check A's recorded runtime

**Run this cell yourself.** On first use it downloads Python/packages into a new Conda environment under this repository. Your existing Anaconda environment is not modified. Setup may take several minutes and needs network access and disk space.

The existing `requirements.txt` pins A's runtime (including PyTorch 2.2.0+cu121, torchvision 0.17.0+cu121 and scikit-learn 1.8.0); Python is pinned to 3.12.5. The worker checks actual versions against A's saved protocol and requires CUDA and Pillow 12.2. A completed setup is checked and reused, not reinstalled. Do not use this cell to modify an environment after final freeze.

In [ ]:
requirements = project / "requirements.txt"
owner_protocol = project / "results" / "round8_a" / "protocol.json"
assert requirements.is_file() and owner_protocol.is_file()
owner = json.loads(owner_protocol.read_text(encoding="utf-8"))["environment"]
setup_receipt = env_prefix.parent / "a_recorded_runtime_setup.json"
setup_identity = {
    "requirements_sha256": hashlib.sha256(requirements.read_bytes()).hexdigest(),
    "owner_protocol_sha256": hashlib.sha256(owner_protocol.read_bytes()).hexdigest(),
    "python": owner["python"],
    "prefix": str(env_prefix),
}
if setup_receipt.exists():
    assert json.loads(setup_receipt.read_text(encoding="utf-8")) == setup_identity, \
        "Setup inputs changed. Do not reinstall into this recorded environment."
    assert (env_prefix / "python.exe").is_file(), "Recorded environment is missing."
else:
    assert not (output / "freeze.json").exists(), "Setup receipt missing after freeze. Stop."
    if not (env_prefix / "python.exe").is_file():
        env_prefix.parent.mkdir(parents=True, exist_ok=True)
        stream([conda, "create", "--yes", "--prefix", str(env_prefix),
                "python=" + owner["python"], "pip"])
    stream(worker_command("-m", "pip", "install", "--requirement", str(requirements)))

# Explicitly check the new interpreter, packages, CUDA and pinned Pillow.
stream(worker_command(str(project / "scripts" / "round8_final_common.py"), "--check-environment"))
if not setup_receipt.exists():
    with setup_receipt.open("x", encoding="utf-8") as handle:
        json.dump(setup_identity, handle, indent=2, sort_keys=True)
        handle.write("\n")
environment_ready = True
print("Recorded runtime check passed. Continue to handoff restoration and reproduction.")

## Preserve the failed development run

Run once before restoring/reproducing. This moves the failed development report and cache into `archive/failed_development/` inside the original results/artifacts folders, with a SHA-256 manifest. File contents are preserved. Completed archival is verified and skipped on later notebook runs.

This cell never resets a successful reproduction or an evaluated test. The same original filenames and active folders are then used by all following cells.

In [ ]:
run("round8_final_common.py", "--archive-failed")

## 1. Verify archives and restore missing fitted model files

Checks both external checksum files, the internal checksum list, all twelve model files and completion receipts. It also verifies the six archived prediction CSVs. Existing files are never replaced. Historical preservation hashing can take several minutes.

In [ ]:
run("round8_final_handoff.py", "--restore-only")

## 2. Prove development reproduction before going further

This cell **does perform development feature extraction on your GPU**. It scores only the existing 900-image development set, under all four conditions, for all six A runs. It does not open the new test set. Numerical differences are printed and saved; failure blocks the remaining stages. Incomplete feature extraction resumes from validated batches.

In [ ]:
reproduction = output / "handoff_reproduction.json"
if reproduction.exists():
    saved = json.loads(reproduction.read_text(encoding="utf-8"))
    assert saved["status"] == "passed", "Prior reproduction failed. Stop and inspect it."
    print("Existing reproduction passed; input hashes are rechecked during freeze.")
else:
    run("round8_final_handoff.py")

saved = json.loads(reproduction.read_text(encoding="utf-8"))
print("Status:", saved["status"])
print("Maximum fused-logit difference:",
      max(r["max_abs_logit"] for r in saved["numerical_differences"]))
print("Maximum probability difference:",
      max(r["max_abs_probability"] for r in saved["numerical_differences"]))
print("Binary-decision disagreements:",
      sum(r["classification_disagreements"] for r in saved["numerical_differences"]))
print("Full report:", reproduction)

## 3. Build the fresh hash-only manifest

Target: 500 each of GLIDE, Midjourney, VQDM, Wukong, ADM and BigGAN, plus 500 shared real images. Every historical pool and its pixel/dHash neighbours are excluded. A shortfall stops the script. No SD15 in-domain test is available.

In [ ]:
if (output / "test_manifest.jsonl").exists():
    run("round8_final_data.py", "--verify")
else:
    run("round8_final_data.py")
    run("round8_final_data.py", "--verify")
print(json.loads((output / "data_audit.json").read_text(encoding="utf-8"))["counts"])

## 4. Freeze models, thresholds, code and test manifest

This cell writes `freeze.json` before final inference. It pins the portable heads, original models, saved gates/weights/thresholds and selected checkpoints. It does not fit or select anything. Notebook outputs can change, but notebook source and scripts must stay unchanged after freezing.

In [ ]:
if (output / "freeze.json").exists():
    run("round8_final_run.py", "--check-freeze")
else:
    run("round8_final_run.py", "--freeze")

## 5. Run the final evaluation once

**This is the final-test GPU inference cell.** It evaluates all 96 declared model instances on original images under clean/JPEG70/JPEG50/resize05. `--resume` continues an interrupted run using saved batches. Once `complete.json` exists, the evaluator rejects further inference; this notebook skips the completed run.

In [ ]:
if (output / "complete.json").exists():
    print("Final evaluation already completed. Skipping inference; continue to verification.")
else:
    run("round8_final_run.py", "--resume")

## 6. Verify and display results

This verifier recomputes metrics from saved predictions, verifies the freeze and historical file hashes, checks exclusions and sampling, and reproduces the summary tables. It does not load or score image models.

In [ ]:
run("round8_final_verify.py", "--report")

from IPython.display import display, Markdown
summary = output / "SUMMARY.md"
display(Markdown(summary.read_text(encoding="utf-8")))
print("All results:", output)
print("Per-seed and mean/SD:", output / "metrics_mean_sd.csv")
print("Within-seed comparisons:", output / "paired_differences.csv")
print("Same model, different test sets:", output / "cross_test_differences.csv")